# Classical.ipynb

## Simple linear regression

Çonsider the line defined by the following equation:
y=ax+b
Here:

- x = Grocery
- y = Delicatessen
- a = slope
- b = intercept

Simple linear regression tests the null hypothesis:
- H0: a=0
- This hypothesis means that the sales of Grocery has nothing to do with that of Delicatessen.

In [1]:
import pandas as pd
import statsmodels.formula.api as smf

df = pd.read_csv("Wholesale customers data.csv")

In [2]:
model = smf.ols('Delicassen ~ Grocery', data=df)
results = model.fit()
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:             Delicassen   R-squared:                       0.042
Model:                            OLS   Adj. R-squared:                  0.040
Method:                 Least Squares   F-statistic:                     19.31
Date:                Mon, 01 Dec 2025   Prob (F-statistic):           1.39e-05
Time:                        14:47:22   Log-Likelihood:                -4109.9
No. Observations:                 440   AIC:                             8224.
Df Residuals:                     438   BIC:                             8232.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   1039.9856    171.831      6.052      0.0

### Conclusion:
The p value is less than 0.05, so H0 is rejected.

## ANOVA

Then, I want to find out whether the average of the sales of Delicatessen varies from region to region.

So, by using ANOVA, I test the null hypothesis:

- H0:There is no significant difference in the average delicatessen expenditure among all regions.

In [3]:
from statsmodels.stats.anova import anova_lm

df = pd.read_csv("Wholesale customers data.csv")

model = smf.ols('Delicassen ~ C(Region)', data=df)
results = model.fit()

anova_table = anova_lm(results)
print(anova_table)

              df        sum_sq       mean_sq         F    PR(>F)
C(Region)    2.0  1.138793e+07  5.693963e+06  0.715022  0.489753
Residual   437.0  3.479978e+09  7.963336e+06       NaN       NaN


### Conclusion:
This result suggests that the hypothesis is not dismissed.

## Two-sample T-tests

### 1. Comparison of Mean Delicassen Expenditure by Frozen Food User Segment

- Analysis Objective: 
The analysis will compare whether there is a difference in the mean delicatessen expenditure between "Heavy Users" and "Middle Users" of Frozen food.

- Definition:
The top 25% of customers (by Frozen expenditure) are designated as the "Heavy Frozen Food Users."
The subsequent 25% (the 50th to 75th percentile) are designated as the "Middle Frozen Food Users."

- Insight: 
This analysis allows us to gain a sharper understanding of customer purchasing strategies and business model trends—specifically, whether customers who purchase a large quantity of frozen foods simultaneously exhibit high spending on value-added, prepared foods like delicatessen items.

This approach highlights subtle differences among the most important customer segments, making it a more crucial business analysis than simply comparing customers who buy "a lot" versus "a little" of frozen foods.

- Null Hypothesis:
The mean expenditure on Delicassen is equal between the Heavy Frozen Food Users and the Middle Frozen Food Users.

In [4]:
from scipy import stats

df = pd.read_csv("Wholesale customers data.csv")

q_50 = df['Frozen'].quantile(0.50)
q_75 = df['Frozen'].quantile(0.75)

heavy_users = df[df['Frozen'] > q_75]['Delicassen']

middle_users = df[(df['Frozen'] > q_50) & (df['Frozen'] <= q_75)]['Delicassen']

t_stat, p_value = stats.ttest_ind(heavy_users, middle_users, equal_var=False)

print(f"--- Two-Sample t-test result ---")
print(f"t-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}")

--- Two-Sample t-test result ---
t-statistic: 2.3601
P-value: 0.0198


### Conclusion:
The p-value is smaller than 0.05.Therefore, the null hypothesis is rejected.This result indicates that Heavy Frozen Food Users have a statistically significantly higher mean annual expenditure on Delicassen compared to Middle Frozen Food Users. This suggests a specific business insight: the purchasing strategy of customers who stock large volumes of frozen food is correlated with higher expenditure on value-added goods, such as delicatessen items.

### 2. Comparing Mean Fresh Spending Between High and Low Grocery Dependence Segments

- Objective: To verify whether customer segments highly dependent on a specific product category exhibit different spending patterns in other categories.

- Segment Creation (Ratio Calculation):Calculate the ratio of Grocery expenditure to the total annual customer spending, and define this as the "Grocery Dependence Ratio."Based on this ratio, segment customers into two groups: the Top 25% Group (High Dependence) and the Bottom 25% Group (Low Dependence).

- Hypothesis Testing (Two-Sample t-test):Test Content: Compare the mean expenditure on Fresh between these two segments.

In [5]:
df = pd.read_csv("Wholesale customers data.csv")

spending_cols = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']
df['Total_Spending'] = df[spending_cols].sum(axis=1)

df['Grocery_Ratio'] = df['Grocery'] / df['Total_Spending']

q_25 = df['Grocery_Ratio'].quantile(0.25)
q_75 = df['Grocery_Ratio'].quantile(0.75)

high_ratio_fresh = df[df['Grocery_Ratio'] > q_75]['Fresh']

low_ratio_fresh = df[df['Grocery_Ratio'] < q_25]['Fresh']

t_stat, p_value = stats.ttest_ind(high_ratio_fresh, low_ratio_fresh, equal_var=False)

print(f"--- Two-Sample t-test result ---")
print(f"t-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}")

--- Two-Sample t-test result ---
t-statistic: -10.2469
P-value: 0.0000


### Conclusion:
The p-value is significantly smaller than 0.05, leading to the rejection of the null hypothesis.This result indicates that the Low Grocery Dependence customer segment has a statistically significantly higher mean expenditure on Fresh products than the High Grocery Dependence segment. This result provides a specific business insight:
A low grocery dependence (meaning a small proportion of total spending is on Grocery) implies that a customer's spending is disproportionately allocated to other categories, such as Fresh, Frozen, and Milk. This segment likely represents Horeca (Hotel, Restaurant, Cafe) channel customers or those prioritizing non-retail-centric sourcing. This clearly confirms a distinct purchasing strategy compared to customers focused primarily on grocery items (retail stores).